# Weather LLM Fine-Tuning: SFT → Reward Model → PPO

This modified notebook runs the pipeline in the intended order: supervised fine-tuning first, then reward model training, then PPO using the SFT adapter as the policy and the reward model as the scorer.

In [ ]:
!pip -q install --upgrade pip
!pip -q install unsloth transformers datasets trl accelerate peft bitsandbytes kagglehub pandas


## Configuration

In [ ]:
MODEL_NAME = "Qwen/Qwen2-7B-Instruct"
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True  # True = QLoRA-style loading. False = LoRA-style loading.

# Output directories
SFT_OUTPUT_DIR = "/content/sft-weather"
SFT_ADAPTER_DIR = f"{SFT_OUTPUT_DIR}-adapter"
RM_OUTPUT_DIR = "/content/weather_reward_model"
RM_ADAPTER_DIR = f"{RM_OUTPUT_DIR}-adapter"
PPO_OUTPUT_DIR = "/content/ppo-weather"
PPO_ADAPTER_DIR = f"{PPO_OUTPUT_DIR}-adapter"

# Training controls
NUM_TRAIN_EPOCHS = 5
RM_NUM_TRAIN_EPOCHS = 2
PPO_NUM_UPDATES = 100          # Keep small at first; increase after confirming stability.
PPO_BATCH_SIZE = 1
PPO_MAX_NEW_TOKENS = 64
PPO_LEARNING_RATE = 5e-6
PPO_CLIP_EPS = 0.2
PPO_VALUE_COEF = 0.5
PPO_ENTROPY_COEF = 0.01
PPO_KL_COEF = 0.02
PPO_GRAD_ACCUM = 1

SYSTEM_PROMPT = "You are a weather forecasting assistant. You should only output in JSON format"
TRAIN_SPLIT = 0.8
FEATURE_COLUMNS = ['date', 'temp_max', 'temp_min', 'precipitation', 'wind', 'weather']
WEATHER_LABELS = ["drizzle", "rain", "sun", "snow", "fog"]
EXTERNAL_DATASET = "seattle-weather.csv"
WINDOW_SIZES = [7, 14, 21, 28]


## Dataset preparation

In [ ]:
from pathlib import Path
import json
import random
from copy import deepcopy
import pandas as pd
import kagglehub

DATA_DIR = Path("/content/weather_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
JSONL_OUT_PATH = DATA_DIR / "seattle_weather_chat.jsonl"
TRAIN_OUT_PATH = DATA_DIR / "seattle_weather_chat_train.jsonl"
EVAL_OUT_PATH = DATA_DIR / "seattle_weather_chat_eval.jsonl"
PREF_TRAIN_OUT_PATH = DATA_DIR / "seattle_weather_pref_train.jsonl"
PREF_EVAL_OUT_PATH = DATA_DIR / "seattle_weather_pref_eval.jsonl"

def format_day(row, day_offset):
    return (
        f"Day {day_offset}: "
        f"temp_max={row['temp_max']}, temp_min={row['temp_min']}, "
        f"precipitation={row['precipitation']}, wind={row['wind']}, "
        f"weather={row['weather']}"
    )

def format_target(row):
    target = {
        "temp_max": float(row['temp_max']),
        "temp_min": float(row['temp_min']),
        "precipitation": float(row['precipitation']),
        "wind": float(row['wind']),
        "weather": row['weather'],
    }
    return json.dumps(target, separators=(",", ":"))

def pull_data(dataset_name, output_directory):
    output_directory.mkdir(parents=True, exist_ok=True)
    return kagglehub.dataset_download(
        dataset_name,
        output_dir=str(output_directory),
        force_download=True,
    )

def format_data(path_to_dataset):
    df = pd.read_csv(path_to_dataset)
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').drop_duplicates(subset=['date']).reset_index(drop=True)
    df = df.dropna().reset_index(drop=True)
    return df

def build_window_samples(data_frame, window_size):
    samples = []
    for t in range(window_size, len(data_frame)):
        window = data_frame.iloc[t - window_size:t]
        target_row = data_frame.iloc[t]

        lines = [format_day(window.iloc[i], i - window_size) for i in range(window_size)]
        user = (
            f"Given the last {window_size} days of Seattle weather, predict tomorrow.\n"
            + "\n".join(lines)
            + "\nReturn JSON with keys: temp_max, temp_min, precipitation, wind, weather."
        )

        samples.append({
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user},
                {"role": "assistant", "content": format_target(target_row)},
            ]
        })
    return samples

def split_samples(samples):
    if len(samples) < 2:
        return samples, []
    split_index = int(len(samples) * TRAIN_SPLIT)
    split_index = min(max(split_index, 1), len(samples) - 1)
    return samples[:split_index], samples[split_index:]

def write_jsonl(path, samples):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as file:
        for sample in samples:
            file.write(json.dumps(sample) + "\n")

def write_samples(data_frame):
    all_samples, train_samples, eval_samples = [], [], []
    for window_size in WINDOW_SIZES:
        window_samples = build_window_samples(data_frame, window_size)
        split_train, split_eval = split_samples(window_samples)
        all_samples.extend(window_samples)
        train_samples.extend(split_train)
        eval_samples.extend(split_eval)

    write_jsonl(JSONL_OUT_PATH, all_samples)
    write_jsonl(TRAIN_OUT_PATH, train_samples)
    write_jsonl(EVAL_OUT_PATH, eval_samples)

    print(f"Total samples: {len(all_samples)}")
    print(f"Train samples: {len(train_samples)}")
    print(f"Eval samples: {len(eval_samples)}")
    return all_samples, train_samples, eval_samples

def make_rejected_response(target_dict, rng, hard_negative=False):
    bad = deepcopy(target_dict)

    if hard_negative:
        # Small realistic mistakes
        bad["temp_max"] = round(float(bad["temp_max"]) + rng.uniform(-2.0, 2.0), 1)
        bad["temp_min"] = round(float(bad["temp_min"]) + rng.uniform(-2.0, 2.0), 1)
        bad["precipitation"] = round(max(0.0, float(bad["precipitation"]) + rng.uniform(-2.0, 2.0)), 1)
        bad["wind"] = round(max(0.0, float(bad["wind"]) + rng.uniform(-1.5, 1.5)), 1)

        if rng.random() < 0.35:
            other_labels = [x for x in WEATHER_LABELS if x != bad["weather"]]
            if other_labels:
                bad["weather"] = rng.choice(other_labels)
    else:
        # Bigger obvious mistakes
        corruption_types = rng.sample(
            ["temp_max", "temp_min", "precipitation", "wind", "weather", "swap_temps"],
            k=rng.randint(1, 3),
        )

        if "temp_max" in corruption_types:
            bad["temp_max"] = round(float(bad["temp_max"]) + rng.uniform(-8.0, 8.0), 1)

        if "temp_min" in corruption_types:
            bad["temp_min"] = round(float(bad["temp_min"]) + rng.uniform(-8.0, 8.0), 1)

        if "precipitation" in corruption_types:
            bad["precipitation"] = round(
                max(0.0, float(bad["precipitation"]) + rng.uniform(-10.0, 10.0)), 1
            )

        if "wind" in corruption_types:
            bad["wind"] = round(max(0.0, float(bad["wind"]) + rng.uniform(-5.0, 5.0)), 1)

        if "weather" in corruption_types:
            other_labels = [x for x in WEATHER_LABELS if x != bad["weather"]]
            if other_labels:
                bad["weather"] = rng.choice(other_labels)

        if "swap_temps" in corruption_types:
            bad["temp_max"], bad["temp_min"] = bad["temp_min"], bad["temp_max"]

    return json.dumps(bad, separators=(",", ":"))

def sft_to_preference_samples(sft_samples, seed=42, include_system=False):
    rng = random.Random(seed)
    preference_samples = []

    for sample in sft_samples:
        messages = sample.get("messages", [])

        system_msg = next((m["content"] for m in messages if m["role"] == "system"), "")
        user_msg = next((m["content"] for m in messages if m["role"] == "user"), None)
        assistant_msg = next((m["content"] for m in messages if m["role"] == "assistant"), None)

        if not user_msg or not assistant_msg:
            continue

        try:
            target_dict = json.loads(assistant_msg)
        except json.JSONDecodeError:
            continue

        prompt = user_msg if not include_system else f"System: {system_msg}\n\nUser: {user_msg}"

        # Easy negative
        rejected_easy = make_rejected_response(target_dict, rng, hard_negative=False)
        if rejected_easy != assistant_msg:
            preference_samples.append({
                "prompt": prompt,
                "chosen": assistant_msg,
                "rejected": rejected_easy,
            })

        # Hard negative
        rejected_hard = make_rejected_response(target_dict, rng, hard_negative=True)
        if rejected_hard != assistant_msg:
            preference_samples.append({
                "prompt": prompt,
                "chosen": assistant_msg,
                "rejected": rejected_hard,
            })

    return preference_samples

def write_preference_samples(train_sft_samples, eval_sft_samples):
    pref_train = sft_to_preference_samples(train_sft_samples, seed=42, include_system=False)
    pref_eval = sft_to_preference_samples(eval_sft_samples, seed=123, include_system=False)

    write_jsonl(PREF_TRAIN_OUT_PATH, pref_train)
    write_jsonl(PREF_EVAL_OUT_PATH, pref_eval)

    print(f"Train preference samples: {len(pref_train)}")
    print(f"Eval preference samples: {len(pref_eval)}")
    return pref_train, pref_eval

weather_data_path = pull_data("ananthr1/weather-prediction", DATA_DIR / "weather-prediction")
df = format_data(Path(weather_data_path) / EXTERNAL_DATASET)
all_samples, train_samples, eval_samples = write_samples(df)
pref_train_samples, pref_eval_samples = write_preference_samples(train_samples, eval_samples)

print(df.head())
print("\nSFT example:")
print(train_samples[0])

print("\nPreference example:")
print(pref_train_samples[0])

## Stage 1: Supervised Fine-Tuning (SFT)

In [ ]:
from datasets import load_dataset
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
)

raw_datasets = load_dataset(
    "json",
    data_files={
        "train": str(TRAIN_OUT_PATH),
        "eval": str(EVAL_OUT_PATH),
    },
)

def apply_template(batch):
    texts = []
    for messages in batch["messages"]:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    return {"text": texts}

dataset = raw_datasets.map(apply_template, batched=True)
dataset


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=67,
    use_rslora=False,
    loftq_config=None,
)


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback
import shutil

sft_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["eval"],
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    args=TrainingArguments(
        output_dir=SFT_OUTPUT_DIR,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        learning_rate=2e-4,
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=67,
        fp16=False,
        bf16=True,
        report_to="none",
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
    ),
)

sft_stats = sft_trainer.train()
print(sft_stats)

model.save_pretrained(SFT_ADAPTER_DIR)
tokenizer.save_pretrained(SFT_ADAPTER_DIR)
shutil.make_archive(SFT_ADAPTER_DIR, "zip", SFT_ADAPTER_DIR)
print(f"Saved SFT adapter to: {SFT_ADAPTER_DIR}")


## Stage 2: Reward Model Training

The reward model is trained after SFT. It learns to score chosen weather JSON outputs higher than corrupted rejected outputs.

In [ ]:
from datasets import load_dataset
from trl import RewardTrainer, RewardConfig
from peft import LoraConfig
from transformers import AutoModelForSequenceClassification, AutoTokenizer, BitsAndBytesConfig
import torch, shutil, gc

# Free the SFT trainer object before loading the reward model.
try:
    del sft_trainer
except NameError:
    pass
if torch.cuda.is_available():
    torch.cuda.empty_cache()

def get_dtype():
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    if torch.cuda.is_available():
        return torch.float16
    return torch.float32

rm_dataset = load_dataset(
    "json",
    data_files={
        "train": str(PREF_TRAIN_OUT_PATH),
        "eval": str(PREF_EVAL_OUT_PATH),
    },
)

rm_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if rm_tokenizer.pad_token is None:
    rm_tokenizer.pad_token = rm_tokenizer.eos_token

rm_load_kwargs = {
    "num_labels": 1,
    "trust_remote_code": False,
}
if LOAD_IN_4BIT:
    rm_load_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=get_dtype(),
    )
    rm_load_kwargs["device_map"] = "auto"
else:
    rm_load_kwargs["torch_dtype"] = get_dtype()

rm_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, **rm_load_kwargs)
rm_model.config.pad_token_id = rm_tokenizer.pad_token_id

rm_peft_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    task_type="SEQ_CLS",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    modules_to_save=["score"],
)

rm_training_args = RewardConfig(
    output_dir=RM_OUTPUT_DIR,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=RM_NUM_TRAIN_EPOCHS,
    learning_rate=1e-4,
    warmup_ratio=0.03,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    report_to="none",
    max_length=MAX_SEQ_LENGTH,
    center_rewards_coefficient=1e-2,
)

rm_trainer = RewardTrainer(
    model=rm_model,
    processing_class=rm_tokenizer,
    args=rm_training_args,
    train_dataset=rm_dataset["train"],
    eval_dataset=rm_dataset["eval"],
    peft_config=rm_peft_config,
)

rm_stats = rm_trainer.train()
print(rm_stats)

rm_trainer.model.save_pretrained(RM_ADAPTER_DIR)
rm_tokenizer.save_pretrained(RM_ADAPTER_DIR)
shutil.make_archive(RM_ADAPTER_DIR, "zip", RM_ADAPTER_DIR)
print("Saved reward-model adapter to:", RM_ADAPTER_DIR)


## Stage 3: PPO Post-Training

This stage loads the SFT adapter as the trainable policy, a frozen SFT adapter as the reference model, and the learned reward-model adapter as the scorer. It applies a clipped PPO-style update to the trainable adapter weights.

In [ ]:
# Simplified PPO stage: SFT adapter policy + frozen SFT reference + learned reward model.
# This is intentionally explicit instead of hiding the PPO loop behind a trainer.

import json, math, random, gc, shutil
from pathlib import Path
from typing import Any

import torch
import torch.nn as nn
import torch.nn.functional as F
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer, BitsAndBytesConfig
from torch.optim import AdamW

# Free reward trainer object but keep the saved adapter path.
try:
    del rm_trainer, rm_model
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


def get_dtype():
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    if torch.cuda.is_available():
        return torch.float16
    return torch.float32


def load_causal_backbone(trainable: bool):
    kwargs = {"trust_remote_code": False}
    if LOAD_IN_4BIT:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=get_dtype(),
        )
        kwargs["device_map"] = "auto"
    else:
        kwargs["torch_dtype"] = get_dtype()
    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **kwargs)
    model = PeftModel.from_pretrained(base, SFT_ADAPTER_DIR, is_trainable=trainable)
    model.config.use_cache = False
    return model

ppo_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if ppo_tokenizer.pad_token is None:
    ppo_tokenizer.pad_token = ppo_tokenizer.eos_token
ppo_tokenizer.padding_side = "left"

policy = load_causal_backbone(trainable=True)
reference = load_causal_backbone(trainable=False)
reference.eval()
for p in reference.parameters():
    p.requires_grad_(False)

# Load reward model as frozen scorer.
rm_kwargs = {"num_labels": 1, "trust_remote_code": False}
if LOAD_IN_4BIT:
    rm_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=get_dtype(),
    )
    rm_kwargs["device_map"] = "auto"
else:
    rm_kwargs["torch_dtype"] = get_dtype()
reward_base = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, **rm_kwargs)
reward_model = PeftModel.from_pretrained(reward_base, RM_ADAPTER_DIR, is_trainable=False)
reward_model.eval()
for p in reward_model.parameters():
    p.requires_grad_(False)

# Value head for PPO. It is saved separately from the adapter.
hidden_size = policy.config.hidden_size
value_head = nn.Linear(hidden_size, 1).to(next(policy.parameters()).device, dtype=get_dtype())

optimizer = AdamW(
    [p for p in policy.parameters() if p.requires_grad] + list(value_head.parameters()),
    lr=PPO_LEARNING_RATE,
)

# Prompt-only PPO dataset from the original SFT training rows.
def build_ppo_prompts(path, max_items=None):
    prompts = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            prompt_messages = [m for m in row["messages"] if m["role"] != "assistant"]
            prompt_text = ppo_tokenizer.apply_chat_template(
                prompt_messages,
                tokenize=False,
                add_generation_prompt=True,
            )
            prompts.append(prompt_text)
            if max_items is not None and len(prompts) >= max_items:
                break
    return prompts

ppo_prompts = build_ppo_prompts(str(TRAIN_OUT_PATH))
random.Random(67).shuffle(ppo_prompts)


def logprobs_from_logits(logits, labels):
    log_probs = logits[:, :-1, :].float().log_softmax(dim=-1)
    return log_probs.gather(-1, labels[:, 1:].unsqueeze(-1)).squeeze(-1)


def masked_mean(x, mask):
    mask = mask.to(x.dtype)
    return (x * mask).sum() / mask.sum().clamp_min(1.0)


def score_with_reward_model(prompts, responses):
    texts = [p + r for p, r in zip(prompts, responses)]
    inputs = rm_tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(next(reward_model.parameters()).device)
    with torch.no_grad():
        scores = reward_model(**inputs).logits.squeeze(-1).float()
    return scores.to(next(policy.parameters()).device)


def get_response_stats(model, input_ids, attention_mask, prompt_width, response_width):
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        output_hidden_states=True,
        return_dict=True,
    )
    token_logprobs = logprobs_from_logits(outputs.logits, input_ids)
    start = max(prompt_width - 1, 0)
    end = start + response_width
    response_logprobs = token_logprobs[:, start:end]
    hidden = outputs.hidden_states[-1][:, :-1, :]
    response_hidden = hidden[:, start:end, :].to(value_head.weight.dtype)
    values = value_head(response_hidden).squeeze(-1).float()
    entropy = -(outputs.logits[:, start:end, :].float().softmax(-1) * outputs.logits[:, start:end, :].float().log_softmax(-1)).sum(-1)
    return response_logprobs.float(), values, entropy.float()


def response_mask_from_ids(response_ids):
    mask = response_ids.ne(ppo_tokenizer.pad_token_id)
    if ppo_tokenizer.eos_token_id is None:
        return mask
    eos_seen = torch.zeros(response_ids.size(0), dtype=torch.bool, device=response_ids.device)
    final_mask = torch.zeros_like(mask)
    for t in range(response_ids.size(1)):
        active = mask[:, t] & ~eos_seen
        final_mask[:, t] = active
        eos_seen = eos_seen | (active & response_ids[:, t].eq(ppo_tokenizer.eos_token_id))
    return final_mask


for update in range(PPO_NUM_UPDATES):
    batch_prompts = ppo_prompts[update * PPO_BATCH_SIZE : (update + 1) * PPO_BATCH_SIZE]
    if not batch_prompts:
        break

    policy.eval()
    prompt_inputs = ppo_tokenizer(
        batch_prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH - PPO_MAX_NEW_TOKENS,
    ).to(next(policy.parameters()).device)
    prompt_width = prompt_inputs["input_ids"].shape[1]

    with torch.no_grad():
        sequence_ids = policy.generate(
            **prompt_inputs,
            max_new_tokens=PPO_MAX_NEW_TOKENS,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=ppo_tokenizer.pad_token_id,
            eos_token_id=ppo_tokenizer.eos_token_id,
        )

    response_ids = sequence_ids[:, prompt_width:]
    response_width = response_ids.shape[1]
    response_mask = response_mask_from_ids(response_ids).float()
    sequence_attention_mask = torch.cat([prompt_inputs["attention_mask"], response_mask.long()], dim=1)
    responses = [ppo_tokenizer.decode(ids[mask.bool()], skip_special_tokens=True).strip()
                 for ids, mask in zip(response_ids, response_mask)]

    with torch.no_grad():
        old_logprobs, old_values, _ = get_response_stats(policy, sequence_ids, sequence_attention_mask, prompt_width, response_width)
        ref_logprobs, _, _ = get_response_stats(reference, sequence_ids, sequence_attention_mask, prompt_width, response_width)
        rm_scores = score_with_reward_model(batch_prompts, responses)

    # Token reward = KL penalty at each generated token + terminal RM reward.
    rewards = -PPO_KL_COEF * (old_logprobs - ref_logprobs) * response_mask
    last_indices = response_mask.long().sum(dim=1) - 1
    for row_idx, last_idx in enumerate(last_indices.tolist()):
        if last_idx >= 0:
            rewards[row_idx, last_idx] += rm_scores[row_idx]

    # One-step/sequence advantage estimate using the value head.
    returns = rewards.sum(dim=1, keepdim=True).expand_as(old_values) * response_mask
    advantages = (returns - old_values) * response_mask
    adv_mean = masked_mean(advantages, response_mask)
    adv_std = torch.sqrt(masked_mean((advantages - adv_mean) ** 2, response_mask) + 1e-8)
    advantages = ((advantages - adv_mean) / adv_std) * response_mask

    policy.train()
    for _ in range(2):
        new_logprobs, new_values, entropy = get_response_stats(policy, sequence_ids, sequence_attention_mask, prompt_width, response_width)
        ratio = torch.exp(torch.clamp(new_logprobs - old_logprobs.detach(), -20, 20))
        unclipped = ratio * advantages.detach()
        clipped = torch.clamp(ratio, 1 - PPO_CLIP_EPS, 1 + PPO_CLIP_EPS) * advantages.detach()
        policy_loss = -masked_mean(torch.minimum(unclipped, clipped), response_mask)
        value_loss = masked_mean((new_values - returns.detach()) ** 2, response_mask)
        entropy_bonus = masked_mean(entropy, response_mask)
        loss = policy_loss + PPO_VALUE_COEF * value_loss - PPO_ENTROPY_COEF * entropy_bonus

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_([p for p in policy.parameters() if p.requires_grad] + list(value_head.parameters()), 1.0)
        optimizer.step()

    if update % 10 == 0:
        print({
            "update": update,
            "reward": float(rm_scores.mean().item()),
            "loss": float(loss.item()),
            "policy_loss": float(policy_loss.item()),
            "value_loss": float(value_loss.item()),
            "mean_response_len": float(response_mask.sum(dim=1).float().mean().item()),
        })

policy.save_pretrained(PPO_ADAPTER_DIR)
ppo_tokenizer.save_pretrained(PPO_ADAPTER_DIR)
torch.save(value_head.state_dict(), Path(PPO_ADAPTER_DIR) / "ppo_value_head.pt")
shutil.make_archive(PPO_ADAPTER_DIR, "zip", PPO_ADAPTER_DIR)
print(f"Saved PPO adapter to: {PPO_ADAPTER_DIR}")
